# 02 – Missing data, merging and grouping (Day 6 project part)

**Before this notebook:** run `python src/clean_cpi.py` from the repo root. It reads the raw file, cleans a copy,
validates it, and writes `data/cleaned/cpi_clean.csv`. Read the script: every step is commented.

**This notebook produces:** a documented missing-data decision, a state-to-region lookup merged into the data,
summary tables, and a Tableau-ready CSV. Cells marked **TODO** are yours.

In [ ]:
import pandas as pd
import numpy as np

CLEAN_PATH = "data/cleaned/cpi_clean.csv"
MONTHS = ["January","February","March","April","May","June","July","August","September","October","November","December"]

cpi = pd.read_csv(CLEAN_PATH, parse_dates=["period"])
cpi["month"] = pd.Categorical(cpi["month"], categories=MONTHS, ordered=True)   # CSV forgets category order
print(cpi.shape)
cpi.head(3)

## Part 1 – Missing data: investigate, decide, document
Three gaps exist in this file. Count each one in code before writing anything down.

In [ ]:
# Gap 1: inflation is blank in 2025
cpi.groupby("year").agg(rows=("inflation", "size"), missing_inflation=("inflation", lambda s: s.isna().sum()))

In [ ]:
# Gap 2: some state/sector combinations have fewer rows than others
counts = cpi.groupby(["state", "sector"], observed=True).size().unstack()
counts[counts.nunique(axis=1) > 1]

In [ ]:
# Gap 3: the 'code' column is blank for which divisions?
cpi[cpi["code"].isna()]["division"].unique()

### TODO – write the decision log
For each gap, write one row in `cleaning_log.md`: **what I found / how many rows / what I did / why / evidence or source**.
Use only what you can show. If the reason is a guess, label it **Hypothesis**.
Remember: the 2025 reason is documented by MoSPI coverage (year-on-year is comparable only from January 2026).
The Chandigarh Rural reason is not verified yet.

Think about this decision too: `derived_mom_pct` (month-on-month change of the index) exists for 2025.
Is it acceptable to use it in a report? What must you say about it?

## Part 2 – Merging: add a region lookup
A **merge** adds columns from one table to another by matching a key (here, `state`), like VLOOKUP but for whole tables.

**Source for the grouping:** the five Zonal Councils (Northern, Central, Eastern, Western, Southern) plus the North Eastern Council
for the eight north-eastern states. Andaman and Nicobar Islands and Lakshadweep belong to no zonal council
(they attend the Southern Zonal Council as special invitees), so they are grouped as "Islands" here.
This is a **project grouping, not an official statistical classification**. Verify it against the Ministry of Home Affairs
page and cite it in your README.

In [ ]:
zones = pd.DataFrame({
    "state": [
        "Chandigarh", "Delhi", "Haryana", "Himachal Pradesh", "Jammu and Kashmir", "Ladakh", "Punjab", "Rajasthan",
        "Chhattisgarh", "Madhya Pradesh", "Uttar Pradesh", "Uttarakhand",
        "Bihar", "Jharkhand", "Odisha", "West Bengal",
        "Goa", "Gujarat", "Maharashtra", "Dadra and Nagar Haveli and Daman and Diu",
        "Andhra Pradesh", "Karnataka", "Kerala", "Puducherry", "Tamil Nadu", "Telangana",
        "Arunachal Pradesh", "Assam", "Manipur", "Meghalaya", "Mizoram", "Nagaland", "Sikkim", "Tripura",
        "Andaman and Nicobar Islands", "Lakshadweep",
    ],
    "region_group": (["Northern"] * 8 + ["Central"] * 4 + ["Eastern"] * 4 + ["Western"] * 4
                     + ["Southern"] * 6 + ["North-Eastern"] * 8 + ["Islands"] * 2),
})
print(len(zones), "rows;", zones["state"].is_unique and "state is unique" or "DUPLICATE states")

In [ ]:
def merge_zones(df, zones):
    """Left-merge the region lookup onto the data and report anything that did not match."""
    m = df.merge(zones, on="state", how="left", validate="many_to_one", indicator=True)
    unmatched = sorted(m.loc[m["_merge"] == "left_only", "state"].unique())
    print("Unmatched state names:", unmatched)
    return m.drop(columns="_merge")

merged = merge_zones(cpi, zones)

### TODO – fix the unmatched names
The first merge will report unmatched names. That is intended: this is the most common real problem when combining files.
1. Compare the unmatched names against the exact spellings in `cpi["state"].unique()`. What differs? (Capital letters? Words? Spaces?)
2. Fix the **zones table** (the reference you control), never the raw data. A tidy way is `zones["state"] = zones["state"].replace({...})`.
3. "All India" will still be unmatched. That is correct, since it is not a state. Decide what its `region_group` should be
   (for example "National") and set it **after** the merge, with a note in your log.

In [ ]:
# TODO: fix zones here


merged = merge_zones(cpi, zones)
still_unmatched = set(merged.loc[merged["region_group"].isna(), "state"]) - {"All India"}
print("PASS" if not still_unmatched else f"FAIL - still unmatched: {sorted(still_unmatched)}")

## Part 3 – Grouping
`groupby` splits the table into groups, applies a calculation to each, and combines the results.

**Worked example.** September 2026, CPI (General), Combined sector, states only. Look at the `count` column:
a group of 2 states and a group of 8 states are not equally reliable, and the mean treats every state equally,
whatever its population. **This is an unweighted mean of states, not a regional inflation rate.**

In [ ]:
sept = merged[(merged["year"] == 2026) & (merged["month"] == "September") & (merged["sector"] == "Combined")
              & (merged["division"] == "CPI (General)") & (~merged["is_aggregate"])]

sept.groupby("region_group")["inflation"].agg(["mean", "median", "min", "max", "count"]).round(2)

### TODO – your grouping tasks
1. **`state_summary`:** for each state, the average CPI (General) inflation Jan–Sep 2026 for each sector, as columns
   Rural / Urban / Combined, plus `rural_minus_urban`. (Hints: filter to year 2026 and division; `groupby(["state", "sector"])`;
   `.unstack()` turns the sector level into columns.)
2. Which 5 states have the largest absolute rural–urban gap? Is your top state from project Q2 among them?
3. **Region comparison:** repeat the worked example for the average of Jan–Sep. Write one observation and one hypothesis,
   and say in one sentence why this mean is not the same as the All India figure.
4. **Rows per group check:** after merging, confirm that `len(merged) == len(cpi)`. Why would this number grow if the lookup had
   duplicate state names? (That is why `validate="many_to_one"` is in the function.)

In [ ]:
# TODO 1
state_summary = None

# TODO 2

# TODO 3

# TODO 4

## Part 4 – Save a Tableau-ready file
Tableau works best with **long format**: one row per observation, one column per attribute. This file already is.

In [ ]:
import os
final = merged.copy()
final.loc[final["state"] == "All India", "region_group"] = "National"   # decision logged in cleaning_log.md
os.makedirs("data/cleaned", exist_ok=True)
final.to_csv("data/cleaned/cpi_tableau_long.csv", index=False)
print(final.shape, final["region_group"].value_counts(dropna=False).to_dict())

## Documentation checklist (do this before committing)
- `data_dictionary.md`: add `month_num`, `period`, `is_aggregate`, `has_inflation`, `derived_mom_pct`, `region_group`.
  For each, say whether it is **published** by MoSPI or **derived/added by you**.
- `cleaning_log.md`: one row per decision in Part 1, plus the merge fixes in Part 2.
- `README.md`: update the status table, cite the source of the region grouping, add one line about running `python src/clean_cpi.py`.